In [18]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [19]:
sentences_df = pd.read_csv("processed_data/sentences.csv")
opinions_df = pd.read_csv("processed_data/opinions.csv")

In [20]:
sentences_df.head()

,review_id,sentence_id,text,char_count,words_count,opinion_count
0,27925,27925:0,Добрый час суток.,17,3,0.0
1,27925,27925:1,Меня зовут Влада!,17,3,0.0
2,27925,27925:2,Я со своим молодым человеком и пару его друзей...,105,18,0.0
3,27925,27925:3,Обслуживание хорошее нас встретил метрдотель и...,93,13,2.0
4,27925,27925:4,"Он был в уютном уголке в конце главного зала, ...",174,28,2.0


In [21]:
opinions_df.head()

,sentence_id,category,polarity,target,from,to,general_category
0,27925:3,SERVICE#GENERAL,positive,Обслуживание,0,12,SERVICE
1,27925:3,SERVICE#GENERAL,positive,метрдотель,34,44,SERVICE
2,27925:4,AMBIENCE#GENERAL,positive,уголке,16,22,AMBIENCE
3,27925:4,AMBIENCE#GENERAL,positive,приглушенный свет,46,63,AMBIENCE
4,27925:6,FOOD#QUALITY,positive,блюд,53,57,FOOD


In [22]:
merged_df = pd.merge(sentences_df, opinions_df, on="sentence_id", how="inner")
merged_df = merged_df[(merged_df["polarity"] != "conflict") & (merged_df["polarity"].notna())]
merged_df

,review_id,sentence_id,text,char_count,words_count,opinion_count,category,polarity,target,from,to,general_category
0,27925,27925:3,Обслуживание хорошее нас встретил метрдотель и...,93,13,2.0,SERVICE#GENERAL,positive,Обслуживание,0,12,SERVICE
1,27925,27925:3,Обслуживание хорошее нас встретил метрдотель и...,93,13,2.0,SERVICE#GENERAL,positive,метрдотель,34,44,SERVICE
2,27925,27925:4,"Он был в уютном уголке в конце главного зала, ...",174,28,2.0,AMBIENCE#GENERAL,positive,уголке,16,22,AMBIENCE
3,27925,27925:4,"Он был в уютном уголке в конце главного зала, ...",174,28,2.0,AMBIENCE#GENERAL,positive,приглушенный свет,46,63,AMBIENCE
4,27925,27925:6,На наш не большой банкетик мы заказали пару пр...,345,57,6.0,FOOD#QUALITY,positive,блюд,53,57,FOOD
...,...,...,...,...,...,...,...,...,...,...,...,...
4084,9955,9955:1,"Впечатления остались только хорошие,интерьер п...",79,7,2.0,AMBIENCE#GENERAL,positive,интерьер,36,44,AMBIENCE
4085,9955,9955:2,"Готовят вкусно и достаточно быстро,на мой взгляд.",49,7,1.0,FOOD#QUALITY,positive,NaN,0,0,FOOD
4086,9955,9955:3,"Обслуживала нас приятная,веселая девушка,успев...",110,14,2.0,SERVICE#GENERAL,positive,девушка,33,40,SERVICE
4087,9955,9955:3,"Обслуживала нас приятная,веселая девушка,успев...",110,14,2.0,FOOD#PRICES,positive,скидку,91,97,FOOD


In [23]:
X = merged_df[["text", "category"]]
y = merged_df["polarity"]

In [24]:
group_shuffle_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

In [25]:
split_iterator = group_shuffle_split.split(X, y, groups=merged_df["review_id"])
train_idx, val_idx = next(split_iterator)
X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

In [26]:
len(X_train) == len(y_train)

True

In [27]:
len(X_val) == len(y_val)

True

In [28]:
X_train["category"].value_counts()

category
SERVICE#GENERAL             838
FOOD#QUALITY                778
AMBIENCE#GENERAL            582
RESTAURANT#GENERAL          550
FOOD#STYLE_OPTIONS          232
DRINKS#QUALITY               70
RESTAURANT#PRICES            56
FOOD#PRICES                  51
RESTAURANT#MISCELLANEOUS     35
DRINKS#STYLE_OPTIONS         33
LOCATION#GENERAL             30
DRINKS#PRICES                13
Name: count, dtype: int64

In [29]:
train_groups = merged_df.iloc[train_idx]["review_id"]
val_groups = merged_df.iloc[val_idx]["review_id"]
len(set(train_groups).intersection(val_groups))

0

In [32]:
dummy_classifier = DummyClassifier(strategy="most_frequent")
dummy_classifier.fit(X_train, y_train)
y_pred = dummy_classifier.predict(X_val)

In [34]:
f1_score(y_val, y_pred, average="macro")

0.28196106028618345

In [35]:
accuracy_score(y_val, y_pred)

0.7329268292682927

In [37]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       153
     neutral       0.00      0.00      0.00        66
    positive       0.73      1.00      0.85       601

    accuracy                           0.73       820
   macro avg       0.24      0.33      0.28       820
weighted avg       0.54      0.73      0.62       820



c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

# Baseline: DummyClassifier
1. Выбирается самый частый класс
2. Валидация: 820 аннотаций
3. Accuracy = 0.7329
4. Macro F1 = 0.2820
Следующим моделям необходимо превзойти этот результат